In [ ]:
import os, requests
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

def generate_x_post(topic: str) -> str:
    prompt = f"""
        You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

        Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
        Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

        Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

        Here's the topic provided by the user for which you need to generate a post:
        <topic>
        {topic}
        </topic>
"""
    payload = {"model": "gpt-4o", "input": prompt}
    response = requests.post(
        "https://api.openai.com/v1/responses",
        json=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        },
    )

    response_text = (
        response.json().get("output", [{}])[0].get("content", [{}])[0].get("text", "")
    )

    return response_text


def main():
    # user input => AI (LLM) to generate X post => output post

    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)
    print("Generated X post")
    print(x_post)


if __name__ == "__main__":
    main()

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


def generate_x_post(topic: str) -> str:
    prompt = f"""
        You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

        Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
        Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

        Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

        Here's the topic provided by the user for which you need to generate a post:
        <topic>
        {topic}
        </topic>
"""
    response = client.responses.create(model="gpt-4o", input=prompt)

    return response.output_text


def main():
    # user input => AI (LLM) to generate X post => output post

    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)
    print("Generated X post")
    print(x_post)


if __name__ == "__main__":
    main()

In [ ]:
import json

# Run "uv sync" to install the below packages
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


def generate_x_post(topic: str) -> str:
    with open("post-examples.json", "r") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
        <example-{i}>
            <topic>
            {example['topic']}
            </topic>

            <generated-post>
            {example['post']}
            </generated-post>
        </example-{i}>
        """

    prompt = f"""
        You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

        Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.
        Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

        Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

        Here's the topic provided by the user for which you need to generate a post:
        <topic>
        {topic}
        </topic>

        Here are some examples of topics and generated posts:
        <examples>
            {examples_str}
        </examples>

        Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
        Don't use the content from the examples!
"""
    response = client.responses.create(model="gpt-4o", input=prompt)

    return response.output_text


def main():
    # user input => AI (LLM) to generate X post => output post

    usr_input = input("What should the post be about? ")
    x_post = generate_x_post(usr_input)
    print("Generated X post")
    print(x_post)


if __name__ == "__main__":
    main()

In [ ]:
import json, requests

# Run "uv sync" to install the below packages
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


def get_website_html(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching the URL {url}: {e}")
        return ""


def extract_core_website_content(html: str) -> str:
    response = client.responses.create(
        # using gpt-4o-mini because it's great for summarization & extraction tasks (and cheap!)
        model="gpt-4o-mini",
        input=f"""
            You are an expert web content extractor. Your task is to extract the core content from a given HTML page.
            The core content should be the main text, excluding navigation, footers, and other non-essential elements like scripts etc.

            Here is the HTML content:
            <html>
            {html}
            </html>

            Please extract the core content and return it as plain text.
        """
    )

    return response.output_text


def summarize_content(content: str) -> str:
    response = client.responses.create(
        model="gpt-4o-mini",
        input=f"""
            You are an expert summarizer. Your task is to summarize the provided content into a concise and clear summary.

            Here is the content to summarize:
            <content>
            {content}
            </content>

            Please provide a brief summary of the main points in the content. Prefer bullet points and avoid unncessary explanations.
        """
    )

    return response.output_text


def generate_x_post(summary: str) -> str:
    with open("post-examples.json", "r") as f:
        examples = json.load(f)

    examples_str = ""
    for i, example in enumerate(examples, 1):
        examples_str += f"""
        <example-{i}>
            <topic>
            {example['topic']}
            </topic>

            <generated-post>
            {example['post']}
            </generated-post>
        </example-{i}>
        """

    prompt = f"""
        You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

        Your task is to generate a post based on a short text summary.
        Your post must be concise and impactful.
        Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

        Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.

        Here's the text summary which you should use to generate the post:
        <summary>
        {summary}
        </summary>

        Here are some examples of topics and generated posts:
        <examples>
            {examples_str}
        </examples>

        Please use the tone, language, structure , and style of the examples provided above to generate a post that is engaging and relevant to the topic provided by the user.
        Don't use the content from the examples!
"""
    response = client.responses.create(
        model="gpt-4o",
        input=prompt
    )

    return response.output_text


def main():
    website_url = input("Website URL: ")
    print("Fetching website HTML...")
    try:
        html_content = get_website_html(website_url)
    except Exception as e:
        print(f"An error occurred while fetching the website: {e}")
        return
    
    if not html_content:
        print("Failed to fetch the website content. Exiting.")
        return
    
    print("---------")
    print("Extracting core content from the website...")
    core_content = extract_core_website_content(html_content)
    print("Extracted core content:")
    print(core_content)

    print("---------")
    print("Summarizing the core content...")
    summary = summarize_content(core_content)
    print("Generated summary:")
    print(summary)
    
    print("---------")
    print("Generating X post based on the summary...")
    x_post = generate_x_post(summary)
    print("Generated X post:")
    print(x_post)


if __name__ == "__main__":
    main()

In [ ]:
import json, sys, os, sqlite3, requests

# Run "uv sync" to install the below packages
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


def setup_database():
    conn = sqlite3.connect("invoices.db")
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS invoices (
            id INTEGER PRIMARY KEY,
            vendor_name TEXT,
            vendor_address TEXT,
            vendor_tax_id TEXT,
            customer_name TEXT,
            customer_address TEXT,
            customer_tax_id TEXT,
            invoice_number TEXT,
            date TEXT,
            total_amount REAL,
            tax REAL
        )
    ''')
    conn.commit()
    return conn


def insert_invoice_data(conn, invoice_data):
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO invoices (
            vendor_name, vendor_address, vendor_tax_id,
            customer_name, customer_address, customer_tax_id,
            invoice_number, "date", total_amount, tax
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        invoice_data.get("vendor", {}).get("name"),
        invoice_data.get("vendor", {}).get("address"),
        invoice_data.get("vendor", {}).get("taxId"),
        invoice_data.get("customer", {}).get("name"),
        invoice_data.get("customer", {}).get("address"),
        invoice_data.get("customer", {}).get("taxId"),
        invoice_data.get("invoiceNumber"),
        invoice_data.get("date"),
        invoice_data.get("totalAmount"),
        invoice_data.get("tax")
    ))
    conn.commit()


def get_pdf_content(pdf_path: str) -> str:
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text


def extract_invoice_details(pdf_content: str) -> dict:
    prompt = f"""
    You are an expert data extractor who excels at analyzing invoices.

    Extract all relevant data from the below invoice content (which was extracted from a PDF document).
    Make sure to capture data like vendor name, date, amount, tax, tax IDs etc.

    <invoice-content>
    {pdf_content}
    </invoice-content>

    Return your response as a JSON object without any extra text or explanation.
"""
    response = requests.post(
        "https://api.openai.com/v1/responses",
        json={
            "model": "gpt-4o-mini",
            "input": prompt,
            "text": {
                "format": {
                    "name": "invoice",
                    "type": "json_schema",
                    "schema": {
                            "type": "object",
                            "properties": {
                                "vendor": {
                                    "type": "object",
                                    "properties": {
                                        "name": {
                                            "type": "string",
                                            "description": "The name of the vendor or company issuing the invoice."
                                        },
                                        "address": {
                                            "type": "string",
                                            "description": "The address of the vendor."
                                        },
                                        "taxId": {
                                            "type": "string",
                                            "description": "The tax identification number of the vendor."
                                        }
                                    },
                                    "additionalProperties": False,
                                    "required": ["name", "address", "taxId"]
                                },
                                "customer": {
                                    "type": "object",
                                    "properties": {
                                        "name": {
                                            "type": "string",
                                            "description": "The name of the customer or client."
                                        },
                                        "address": {
                                            "type": "string",
                                            "description": "The address of the customer."
                                        },
                                        "taxId": {
                                            "type": "string",
                                            "description": "The tax identification number of the customer."
                                        }
                                    },
                                    "additionalProperties": False,
                                    "required": ["name", "address", "taxId"]
                                },
                                "invoiceNumber": {
                                    "type": "string",
                                    "description": "The unique identifier for the invoice."
                                },
                                "date": {
                                    "type": "string",
                                    "description": "The date when the invoice was issued."
                                },
                                "totalAmount": {
                                    "type": "number",
                                    "description": "The total amount due on the invoice."
                                },
                                "tax": {
                                    "type": "number",
                                    "description": "The total tax amount applied to the invoice."
                                },
                            },
                        "additionalProperties": False,
                        "required": ["vendor", "customer", "invoiceNumber", "date", "totalAmount", "tax"],
                    },
                    "strict": True
                },
            }
        },
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        }
    )

    response.raise_for_status()
    received_json = response.json().get("output", [{}])[0].get(
        "content", [{}])[0].get("text", "{}")
    return json.loads(received_json)


def main():
    if len(sys.argv) < 2:
        print("Usage: python main.py /path/to/file_or_folder")
        return

    path = sys.argv[1]
    pdf_files = []

    if not os.path.exists(path):
        print(f"Error: The path '{path}' does not exist.")
        return

    if os.path.isfile(path):
        if path.lower().endswith(".pdf"):
            pdf_files.append(path)
        else:
            print(f"Error: The file '{path}' is not a PDF file.")
            return
    elif os.path.isdir(path):
        for filename in os.listdir(path):
            if filename.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(path, filename))

    if not pdf_files:
        print("No PDF files found.")
        return

    conn = setup_database()

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            pdf_content = get_pdf_content(pdf_file)
            invoice_details = extract_invoice_details(pdf_content)
            insert_invoice_data(conn, invoice_details)
            print("Extracted Invoice Details:")
            print(invoice_details)
            print("---------")
        except Exception as e:
            print(f"An error occurred while processing {pdf_file}: {e}")

    conn.close()


if __name__ == "__main__":
    main()

In [ ]:
import sys, os, sqlite3

# Run "uv sync" to install the below packages
from pypdf import PdfReader
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI()


class Vendor(BaseModel):
    name: str = Field(...,
                    description="The name of the vendor or company issuing the invoice.")
    address: str = Field(..., description="The address of the vendor.")
    taxId: str = Field(...,
                    description="The tax identification number of the vendor.")


class Customer(BaseModel):
    name: str = Field(..., description="The name of the customer or client.")
    address: str = Field(..., description="The address of the customer.")
    taxId: str = Field(...,
                        description="The tax identification number of the customer.")


class Invoice(BaseModel):
    vendor: Vendor = Field(...,
                            description="Details of the vendor issuing the invoice.")
    customer: Customer = Field(...,
                                description="Details of the customer receiving the invoice.")
    invoiceNumber: str = Field(...,
                                description="Unique identifier for the invoice.")
    date: str = Field(..., description="Date when the invoice was issued.")
    totalAmount: float = Field(...,
                                description="Total amount due on the invoice.")
    tax: float = Field(...,
                        description="Total tax amount applied to the invoice.")


def setup_database():
    conn = sqlite3.connect("invoices.db")
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS invoices (
            id INTEGER PRIMARY KEY,
            vendor_name TEXT,
            vendor_address TEXT,
            vendor_tax_id TEXT,
            customer_name TEXT,
            customer_address TEXT,
            customer_tax_id TEXT,
            invoice_number TEXT,
            date TEXT,
            total_amount REAL,
            tax REAL
        )
    ''')
    conn.commit()
    return conn


def insert_invoice_data(conn, invoice_data):
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO invoices (
            vendor_name, vendor_address, vendor_tax_id,
            customer_name, customer_address, customer_tax_id,
            invoice_number, "date", total_amount, tax
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        invoice_data.get("vendor", {}).get("name"),
        invoice_data.get("vendor", {}).get("address"),
        invoice_data.get("vendor", {}).get("taxId"),
        invoice_data.get("customer", {}).get("name"),
        invoice_data.get("customer", {}).get("address"),
        invoice_data.get("customer", {}).get("taxId"),
        invoice_data.get("invoiceNumber"),
        invoice_data.get("date"),
        invoice_data.get("totalAmount"),
        invoice_data.get("tax")
    ))
    conn.commit()


def get_pdf_content(pdf_path: str) -> str:
    with open(pdf_path, "rb") as f:
        reader = PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text


def extract_invoice_details(pdf_content: str) -> Invoice:
    prompt = f"""
    You are an expert data extractor who excels at analyzing invoices.

    Extract all relevant data from the below invoice content (which was extracted from a PDF document).
    Make sure to capture data like vendor name, date, amount, tax, tax IDs etc.

    <invoice-content>
    {pdf_content}
    </invoice-content>

    Return your response as a JSON object without any extra text or explanation.
"""
    response = client.responses.parse(
        model="gpt-4o-mini",
        input=prompt,
        text_format=Invoice,
    )

    return response.output_parsed


def main():
    if len(sys.argv) < 2:
        print("Usage: python main.py /path/to/file_or_folder")
        return

    path = sys.argv[1]
    pdf_files = []

    if not os.path.exists(path):
        print(f"Error: The path '{path}' does not exist.")
        return

    if os.path.isfile(path):
        if path.lower().endswith(".pdf"):
            pdf_files.append(path)
        else:
            print(f"Error: The file '{path}' is not a PDF file.")
            return
    elif os.path.isdir(path):
        for filename in os.listdir(path):
            if filename.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(path, filename))

    if not pdf_files:
        print("No PDF files found.")
        return

    conn = setup_database()

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            pdf_content = get_pdf_content(pdf_file)
            invoice = extract_invoice_details(pdf_content)
            insert_invoice_data(conn, invoice.model_dump())
            print("Extracted Invoice Details:")
            print(invoice.model_dump())
            print("---------")
        except Exception as e:
            print(f"An error occurred while processing {pdf_file}: {e}")

    conn.close()


if __name__ == "__main__":
    main()

In [ ]:
import sys, os
import os
# Run "uv sync" to install the below packages
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()


def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, 'w', encoding='utf-8') as file:
        file.write(content)


def generate_article_draft(outline: str) -> str:
    print("Generating article draft...")
    example_posts_path = "example_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(
            f"The directory '{example_posts_path}' does not exist.")

    example_posts = []
    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".md") or filename.lower().endswith(".mdx"):
            with open(os.path.join(example_posts_path, filename), 'r', encoding='utf-8') as file:
                example_posts.append(file.read())

    if not example_posts:
        raise ValueError(
            "No example blog posts found in the 'example_posts' directory.")

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.
                """
            },
            {
                "role": "user",
                "content": f"""
                    Write a detailed blog post based on the following outline:

                    <outline>
                    {outline}
                    </outline>

                    Below are some example blog posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                    DON'T use the content from those example posts!

                    Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                    Don't add any additional text or explanations, just return the raw markdown content.
                """
            }
        ]
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]

    outline = load_file(outline_file)

    blog_post_draft = generate_article_draft(outline)

    output_file = outline_file.replace(".txt", "_draft.md")

    save_file(output_file, blog_post_draft)

    print(f"Blog post draft saved to '{output_file}'.")


if __name__ == "__main__":
    main()

In [ ]:
import base64, sys, os
# Run "uv sync" to install the below packages
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

print(os.getenv("OPENAI_API_KEY"))

client = OpenAI()


def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, 'w', encoding='utf-8') as file:
        file.write(content)


def generate_article_draft(outline: str, existing_draft: str | None = None, feedback: str | None = None) -> str:
    print("Generating article draft...")
    example_posts_path = "example_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(
            f"The directory '{example_posts_path}' does not exist.")

    example_posts = []
    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".md") or filename.lower().endswith(".mdx"):
            with open(os.path.join(example_posts_path, filename), 'r', encoding='utf-8') as file:
                example_posts.append(file.read())

    if not example_posts:
        raise ValueError(
            "No example blog posts found in the 'example_posts' directory.")

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    prompt = f"""
                Write a detailed blog post based on the following outline:

                <outline>
                {outline}
                </outline>

                Below are some example blog posts I wrote in the past:
                <example-posts>
                {example_posts_str}
                </example-posts>

                Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                DON'T use the content from those example posts!

                Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                Don't add any additional text or explanations, just return the raw markdown content.
            """

    if existing_draft and feedback:
        example_posts_str += f"\n\n<existing-draft>\n{existing_draft}\n</existing-draft>"
        example_posts_str += f"\n\n<feedback>\n{feedback}\n</feedback>"

        prompt = f"""
            Write an improved version of the following blog post draft:

            <existing-draft>
            {existing_draft}
            </existing-draft>

            The following feedback should be taken into account when writing the improved draft:

            <feedback>
            {feedback}
            </feedback>

            The original draft AND your improved version should be based on the following outline:

            <outline>
            {outline}
            </outline>

            Below are some example blog posts I wrote in the past:
            <example-posts>
            {example_posts_str}
            </example-posts>

            Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
            DON'T use the content from those example posts!

            Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
            Don't add any additional text or explanations, just return the raw markdown content.
        """

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.

                    If the user provides an existing draft, use it in conjunction with the provided outline and feedback to create an improved draft.
                """
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


class Evaluation(BaseModel):
    needs_improvement: bool = Field(
        description="Whether the draft needs to be improved")
    feedback: str = Field(description="Feedback on how to improve the draft")


def evaluate_article_draft(draft: str) -> Evaluation:
    print("Evaluating article draft...")
    response = client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post evaluator.
                    Your job is to evaluate the quality of the blog post draft and provide feedback on how to improve it.
                    The blog post draft is in raw markdown format and based on an outline provided by the user.
                    Take that outline into account when evaluating the draft.
                    The draft should be evaluated based on the following criteria:
                    - Is the draft informative and easy to read?
                    - Is the draft structured and well-organized?
                    - Is the draft written in a clear, concise, and engaging style?
                    - Is the draft written in a way that is easy to understand for both technical and non-technical readers?
                """
            },
            {
                "role": "user",
                "content": f"""
                    Evaluate the following blog post draft:
                    <draft>
                    {draft}
                    </draft>

                    Return the feedback as JSON, indicating whether the draft needs to be improved and why.
                """
            }
        ],
        text_format=Evaluation
    )

    return response.output_parsed


def generate_thumbnail(article: str) -> bytes:
    print("Generating thumbnail...")

    response = client.images.generate(
        model="gpt-image-1",
        prompt=f"Generate a thumbnail for the following blog post: {article}",
        n=1,
        output_format="jpeg",
        size="1536x1024"
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return image_bytes


def generate_linkedin_post(article: str) -> str:
    print("Generating LinkedIn post...")

    example_posts_path = "example_linkedin_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(
            f"The directory '{example_posts_path}' does not exist.")

    example_posts = []

    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".txt"):
            with open(os.path.join(example_posts_path, filename), 'r', encoding='utf-8') as file:
                example_posts.append(file.read())

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    response = client.responses.create(

        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert LinkedIn post generator.
                    Your job is to generate a LinkedIn post for a blog post.
                    The blog post is in raw markdown format.
                    The LinkedIn post should be a single post that captures the essence of the blog post.
                    It should be informative and engaging.
                """
            },
            {
                "role": "user",
                "content": f"""
                    Generate a LinkedIn post for the following blog post:
                    <article>
                    {article}
                    </article>

                    Here are some example LinkedIn posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your LinkedIn post.
                    DON'T use the content from those example posts!
                """
            }
        ]
    )

    return response.output_text


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]
    outline = load_file(outline_file)

    blog_post_draft = generate_article_draft(outline)
    print("Generated blog post draft:")
    print(blog_post_draft)

    thumbnail_image = generate_thumbnail(blog_post_draft)
    thumbnail_file = outline_file.replace(".txt", "_thumbnail.jpeg")
    with open(thumbnail_file, "wb") as f:
        f.write(thumbnail_image)
    print(f"Thumbnail saved to '{thumbnail_file}'.")

    output_file = outline_file.replace(".txt", "_draft.md")
    save_file(output_file, blog_post_draft)
    print(f"Blog post draft saved to '{output_file}'.")


if __name__ == "__main__":
    main()

In [ ]:
import base64, sys, os, concurrent.futures

# Run "uv sync" to install the below packages
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

print(os.getenv("OPENAI_API_KEY"))

client = OpenAI()


def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, "w", encoding="utf-8") as file:
        file.write(content)


def generate_article_draft(
    outline: str, existing_draft: str | None = None, feedback: str | None = None
) -> str:
    print("Generating article draft...")
    example_posts_path = "example_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(f"The directory '{example_posts_path}' does not exist.")

    example_posts = []
    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".md") or filename.lower().endswith(".mdx"):
            with open(
                os.path.join(example_posts_path, filename), "r", encoding="utf-8"
            ) as file:
                example_posts.append(file.read())

    if not example_posts:
        raise ValueError(
            "No example blog posts found in the 'example_posts' directory."
        )

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    prompt = f"""
                Write a detailed blog post based on the following outline:

                <outline>
                {outline}
                </outline>

                Below are some example blog posts I wrote in the past:
                <example-posts>
                {example_posts_str}
                </example-posts>

                Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                DON'T use the content from those example posts!

                Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                Don't add any additional text or explanations, just return the raw markdown content.
            """

    if existing_draft and feedback:
        example_posts_str += (
            f"\n\n<existing-draft>\n{existing_draft}\n</existing-draft>"
        )
        example_posts_str += f"\n\n<feedback>\n{feedback}\n</feedback>"

        prompt = f"""
            Write an improved version of the following blog post draft:

            <existing-draft>
            {existing_draft}
            </existing-draft>

            The following feedback should be taken into account when writing the improved draft:

            <feedback>
            {feedback}
            </feedback>

            The original draft AND your improved version should be based on the following outline:

            <outline>
            {outline}
            </outline>

            Below are some example blog posts I wrote in the past:
            <example-posts>
            {example_posts_str}
            </example-posts>

            Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
            DON'T use the content from those example posts!

            Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
            Don't add any additional text or explanations, just return the raw markdown content.
        """

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.

                    If the user provides an existing draft, use it in conjunction with the provided outline and feedback to create an improved draft.
                """,
            },
            {"role": "user", "content": prompt},
        ],
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


class Evaluation(BaseModel):
    needs_improvement: bool = Field(
        description="Whether the draft needs to be improved"
    )
    feedback: str = Field(description="Feedback on how to improve the draft")


def evaluate_article_draft(draft: str) -> Evaluation:
    print("Evaluating article draft...")
    response = client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post evaluator.
                    Your job is to evaluate the quality of the blog post draft and provide feedback on how to improve it.
                    The blog post draft is in raw markdown format and based on an outline provided by the user.
                    Take that outline into account when evaluating the draft.
                    The draft should be evaluated based on the following criteria:
                    - Is the draft informative and easy to read?
                    - Is the draft structured and well-organized?
                    - Is the draft written in a clear, concise, and engaging style?
                    - Is the draft written in a way that is easy to understand for both technical and non-technical readers?
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Evaluate the following blog post draft:
                    <draft>
                    {draft}
                    </draft>

                    Return the feedback as JSON, indicating whether the draft needs to be improved and why.
                """,
            },
        ],
        text_format=Evaluation,
    )

    return response.output_parsed


def generate_thumbnail(article: str) -> bytes:
    print("Generating thumbnail...")

    response = client.images.generate(
        model="gpt-image-1",
        prompt=f"Generate a thumbnail for the following blog post: {article}",
        n=1,
        output_format="jpeg",
        size="1536x1024",
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return image_bytes


def generate_linkedin_post(article: str) -> str:
    print("Generating LinkedIn post...")

    example_posts_path = "example_linkedin_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(f"The directory '{example_posts_path}' does not exist.")

    example_posts = []

    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".txt"):
            with open(
                os.path.join(example_posts_path, filename), "r", encoding="utf-8"
            ) as file:
                example_posts.append(file.read())

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert LinkedIn post generator.
                    Your job is to generate a LinkedIn post for a blog post.
                    The blog post is in raw markdown format.
                    The LinkedIn post should be a single post that captures the essence of the blog post.
                    It should be informative and engaging.
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Generate a LinkedIn post for the following blog post:
                    <article>
                    {article}
                    </article>

                    Here are some example LinkedIn posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your LinkedIn post.
                    DON'T use the content from those example posts!
                """,
            },
        ],
    )

    return response.output_text


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]
    outline = load_file(outline_file)

    blog_post_draft = None
    cycles = 0
    evaluation = None

    while True:
        blog_post_draft = generate_article_draft(
            outline,
            existing_draft=blog_post_draft,
            feedback=evaluation.feedback if evaluation else None,
        )
        print("Generated blog post draft:")
        print(blog_post_draft)

        evaluation = evaluate_article_draft(blog_post_draft)
        print("Evaluation result:")
        print(evaluation)

        cycles += 1

        if not evaluation.needs_improvement or cycles > 3:
            break

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_thumbnail = executor.submit(generate_thumbnail, blog_post_draft)
        future_linkedin = executor.submit(generate_linkedin_post, blog_post_draft)

        linkedin_post = future_linkedin.result()
        thumbnail_image = future_thumbnail.result()

    thumbnail_file = outline_file.replace(".txt", "_thumbnail.jpeg")
    with open(thumbnail_file, "wb") as f:
        f.write(thumbnail_image)
    print(f"Thumbnail saved to '{thumbnail_file}'.")

    linkedin_post_file = outline_file.replace(".txt", "_linkedin_post.txt")
    save_file(linkedin_post_file, linkedin_post)
    print(f"LinkedIn post saved to '{linkedin_post_file}'.")

    output_file = outline_file.replace(".txt", "_draft.md")
    save_file(output_file, blog_post_draft)
    print(f"Blog post draft saved to '{output_file}'.")


if __name__ == "__main__":
    main()

In [ ]:
import base64, sys, os, concurrent.futures

# Run "uv sync" to install the below packages
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

client = OpenAI()


def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, 'r', encoding='utf-8') as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, 'w', encoding='utf-8') as file:
        file.write(content)


def generate_article_draft(outline: str, existing_draft: str | None = None, feedback: str | None = None) -> str:
    print("Generating article draft...")
    example_posts_path = "example_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(
            f"The directory '{example_posts_path}' does not exist.")

    example_posts = []
    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".md") or filename.lower().endswith(".mdx"):
            with open(os.path.join(example_posts_path, filename), 'r', encoding='utf-8') as file:
                example_posts.append(file.read())

    if not example_posts:
        raise ValueError(
            "No example blog posts found in the 'example_posts' directory.")

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    prompt = f"""
                Write a detailed blog post based on the following outline:

                <outline>
                {outline}
                </outline>

                Below are some example blog posts I wrote in the past:
                <example-posts>
                {example_posts_str}
                </example-posts>

                Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                DON'T use the content from those example posts!

                Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                Don't add any additional text or explanations, just return the raw markdown content.
            """

    if existing_draft and feedback:
        example_posts_str += f"\n\n<existing-draft>\n{existing_draft}\n</existing-draft>"
        example_posts_str += f"\n\n<feedback>\n{feedback}\n</feedback>"

        prompt = f"""
            Write an improved version of the following blog post draft:

            <existing-draft>
            {existing_draft}
            </existing-draft>

            The following feedback should be taken into account when writing the improved draft:

            <feedback>
            {feedback}
            </feedback>

            The original draft AND your improved version should be based on the following outline:

            <outline>
            {outline}
            </outline>

            Below are some example blog posts I wrote in the past:
            <example-posts>
            {example_posts_str}
            </example-posts>

            Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
            DON'T use the content from those example posts!

            Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
            Don't add any additional text or explanations, just return the raw markdown content.
        """

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.

                    If the user provides an existing draft, use it in conjunction with the provided outline and feedback to create an improved draft.
                """
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


class Evaluation(BaseModel):
    needs_improvement: bool = Field(
        description="Whether the draft needs to be improved")
    feedback: str = Field(description="Feedback on how to improve the draft")


def evaluate_article_draft(draft: str) -> Evaluation:
    print("Evaluating article draft...")
    response = client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post evaluator.
                    Your job is to evaluate the quality of the blog post draft and provide feedback on how to improve it.
                    The blog post draft is in raw markdown format and based on an outline provided by the user.
                    Take that outline into account when evaluating the draft.
                    The draft should be evaluated based on the following criteria:
                    - Is the draft informative and easy to read?
                    - Is the draft structured and well-organized?
                    - Is the draft written in a clear, concise, and engaging style?
                    - Is the draft written in a way that is easy to understand for both technical and non-technical readers?
                """
            },
            {
                "role": "user",
                "content": f"""
                    Evaluate the following blog post draft:
                    <draft>
                    {draft}
                    </draft>

                    Return the feedback as JSON, indicating whether the draft needs to be improved and why.
                """
            }
        ],
        text_format=Evaluation
    )

    return response.output_parsed


def generate_thumbnail(article: str) -> bytes:
    print("Generating thumbnail...")

    response = client.images.generate(
        model="gpt-image-1",
        prompt=f"Generate a thumbnail for the following blog post: {article}",
        n=1,
        output_format="jpeg",
        size="1536x1024"
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return image_bytes


def generate_linkedin_post(article: str) -> str:
    print("Generating LinkedIn post...")

    example_posts_path = "example_linkedin_posts"

    if not os.path.exists(example_posts_path):
        raise FileNotFoundError(
            f"The directory '{example_posts_path}' does not exist.")

    example_posts = []

    for filename in os.listdir(example_posts_path):
        if filename.lower().endswith(".txt"):
            with open(os.path.join(example_posts_path, filename), 'r', encoding='utf-8') as file:
                example_posts.append(file.read())

    example_posts_str = "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )

    response = client.responses.create(

        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert LinkedIn post generator.
                    Your job is to generate a LinkedIn post for a blog post.
                    The blog post is in raw markdown format.
                    The LinkedIn post should be a single post that captures the essence of the blog post.
                    It should be informative and engaging.
                """
            },
            {
                "role": "user",
                "content": f"""
                    Generate a LinkedIn post for the following blog post:
                    <article>
                    {article}
                    </article>

                    Here are some example LinkedIn posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your LinkedIn post.
                    DON'T use the content from those example posts!
                """
            }
        ]
    )

    return response.output_text


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]
    outline = load_file(outline_file)

    blog_post_draft = None
    cycles = 0
    evaluation = None

    while True:
        blog_post_draft = generate_article_draft(
            outline,
            existing_draft=blog_post_draft,
            feedback=evaluation.feedback if evaluation else None
        )
        print("Generated blog post draft:")
        print(blog_post_draft)

        evaluation = evaluate_article_draft(blog_post_draft)
        print("Evaluation result:")
        print(evaluation)

        print("Your feedback on the article:")
        print("--------------------------------")
        print("Hit ENTER to accept the evaluation results without any changes.")
        print("Type 'accept' to accept the article as is (and overwrite any suggested changes).")
        print("Otherwise enter your feedback to improve the article.")
        print("--------------------------------")
        user_feedback = input("Your feedback: ")

        if user_feedback.strip().lower() == "accept":
            evaluation.needs_improvement = False
        elif user_feedback.strip():
            evaluation.needs_improvement = True
            evaluation.feedback = user_feedback

        cycles += 1

        if not evaluation.needs_improvement or (cycles > 3 and not user_feedback.strip()):
            break

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_thumbnail = executor.submit(generate_thumbnail, blog_post_draft)
        future_linkedin = executor.submit(
            generate_linkedin_post, blog_post_draft)

        linkedin_post = future_linkedin.result()
        thumbnail_image = future_thumbnail.result()

    thumbnail_file = outline_file.replace(".txt", "_thumbnail.jpeg")
    with open(thumbnail_file, "wb") as f:
        f.write(thumbnail_image)
    print(f"Thumbnail saved to '{thumbnail_file}'.")

    linkedin_post_file = outline_file.replace(".txt", "_linkedin_post.txt")
    save_file(linkedin_post_file, linkedin_post)
    print(f"LinkedIn post saved to '{linkedin_post_file}'.")

    output_file = outline_file.replace(".txt", "_draft.md")
    save_file(output_file, blog_post_draft)
    print(f"Blog post draft saved to '{output_file}'.")


if __name__ == "__main__":
    main()

In [ ]:
import base64, sys, os, requests, concurrent.futures
# Run "uv sync" to install the below packages
import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

client = OpenAI()

def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, "w", encoding="utf-8") as file:
        file.write(content)


def load_and_format_example_posts(
    path: str, allowed_extensions: list[str], required: bool = False
) -> str:
    """Loads and formats example posts from a directory."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"The directory '{path}' does not exist.")

    example_posts = []
    for filename in os.listdir(path):
        if any(filename.lower().endswith(ext) for ext in allowed_extensions):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
                example_posts.append(file.read())

    if required and not example_posts:
        raise ValueError(
            f"No example posts with extensions {allowed_extensions} found in the '{path}' directory."
        )

    return "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )


def generate_article_draft(
    outline: str, existing_draft: str | None = None, feedback: str | None = None
) -> str:
    print("Generating article draft...")
    example_posts_str = load_and_format_example_posts(
        "example_posts", [".md", ".mdx"], required=True
    )

    prompt = f"""
                Write a detailed blog post based on the following outline:

                <outline>
                {outline}
                </outline>

                Below are some example blog posts I wrote in the past:
                <example-posts>
                {example_posts_str}
                </example-posts>

                Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                DON'T use the content from those example posts!

                Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                Don't add any additional text or explanations, just return the raw markdown content.
            """

    if existing_draft and feedback:
        example_posts_str += (
            f"\n\n<existing-draft>\n{existing_draft}\n</existing-draft>"
        )
        example_posts_str += f"\n\n<feedback>\n{feedback}\n</feedback>"

        prompt = f"""
            Write an improved version of the following blog post draft:

            <existing-draft>
            {existing_draft}
            </existing-draft>

            The following feedback should be taken into account when writing the improved draft:

            <feedback>
            {feedback}
            </feedback>

            The original draft AND your improved version should be based on the following outline:

            <outline>
            {outline}
            </outline>

            Below are some example blog posts I wrote in the past:
            <example-posts>
            {example_posts_str}
            </example-posts>

            Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
            DON'T use the content from those example posts!

            Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
            Don't add any additional text or explanations, just return the raw markdown content.
        """

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.

                    If the user provides an existing draft, use it in conjunction with the provided outline and feedback to create an improved draft.
                """,
            },
            {"role": "user", "content": prompt},
        ],
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


class Evaluation(BaseModel):
    needs_improvement: bool = Field(
        description="Whether the draft needs to be improved"
    )
    feedback: str = Field(description="Feedback on how to improve the draft")


def evaluate_article_draft(draft: str) -> Evaluation:
    print("Evaluating article draft...")
    response = client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post evaluator.
                    Your job is to evaluate the quality of the blog post draft and provide feedback on how to improve it.
                    The blog post draft is in raw markdown format and based on an outline provided by the user.
                    Take that outline into account when evaluating the draft.
                    The draft should be evaluated based on the following criteria:
                    - Is the draft informative and easy to read?
                    - Is the draft structured and well-organized?
                    - Is the draft written in a clear, concise, and engaging style?
                    - Is the draft written in a way that is easy to understand for both technical and non-technical readers?
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Evaluate the following blog post draft:
                    <draft>
                    {draft}
                    </draft>

                    Return the feedback as JSON, indicating whether the draft needs to be improved and why.
                """,
            },
        ],
        text_format=Evaluation,
    )

    return response.output_parsed


def generate_thumbnail(article: str) -> bytes:
    print("Generating thumbnail...")

    response = client.images.generate(
        model="gpt-image-1",
        prompt=f"Generate a thumbnail for the following blog post: {article}",
        n=1,
        output_format="jpeg",
        size="1536x1024",
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return image_bytes


def generate_linkedin_post(article: str) -> str:
    print("Generating LinkedIn post...")

    example_posts_str = load_and_format_example_posts(
        "example_linkedin_posts", [".txt"]
    )

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert LinkedIn post generator.
                    Your job is to generate a LinkedIn post for a blog post.
                    The LinkedIn post should be a single post that captures the essence of the blog post.
                    It should be informative and engaging.
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Generate a LinkedIn post for the following blog post:
                    <article>
                    {article}
                    </article>

                    Here are some example LinkedIn posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your LinkedIn post.
                    DON'T use the content from those example posts!
                """,
            },
        ],
    )

    return response.output_text


def send_slack_notification(message: str) -> None:
    print("Sending Slack notification...")
    response = requests.post(
        "https://slack.com/api/chat.postMessage",
        json={"channel": "C092YQF30BC", "text": message},
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ.get('SLACK_ACCESS_TOKEN')}",
        },
    )

    response.raise_for_status()
    data = response.json()

    if data.get("ok"):
        print("Slack notification sent successfully.")
    else:
        raise Exception(f"Failed to send Slack notification: {data.get('error')}")


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]
    outline = load_file(outline_file)

    blog_post_draft = None
    cycles = 0
    evaluation = None

    while True:
        blog_post_draft = generate_article_draft(
            outline,
            existing_draft=blog_post_draft,
            feedback=evaluation.feedback if evaluation else None,
        )
        print("Generated blog post draft:")
        print(blog_post_draft)

        evaluation = evaluate_article_draft(blog_post_draft)
        print("Evaluation result:")
        print(evaluation)

        print("Your feedback on the article:")
        print("--------------------------------")
        print("Hit ENTER to accept the evaluation results without any changes.")
        print(
            "Type 'accept' to accept the article as is (and overwrite any suggested changes)."
        )
        print("Otherwise enter your feedback to improve the article.")
        print("--------------------------------")
        user_feedback = input("Your feedback: ")

        if user_feedback.strip().lower() == "accept":
            evaluation.needs_improvement = False
        elif user_feedback.strip():
            evaluation.needs_improvement = True
            evaluation.feedback = user_feedback

        cycles += 1

        if not evaluation.needs_improvement or (
            cycles > 3 and not user_feedback.strip()
        ):
            break

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_thumbnail = executor.submit(generate_thumbnail, blog_post_draft)
        future_linkedin = executor.submit(generate_linkedin_post, blog_post_draft)

        linkedin_post = future_linkedin.result()
        thumbnail_image = future_thumbnail.result()

    thumbnail_file = outline_file.replace(".txt", "_thumbnail.jpeg")
    with open(thumbnail_file, "wb") as f:
        f.write(thumbnail_image)
    print(f"Thumbnail saved to '{thumbnail_file}'.")

    linkedin_post_file = outline_file.replace(".txt", "_linkedin_post.txt")
    save_file(linkedin_post_file, linkedin_post)
    print(f"LinkedIn post saved to '{linkedin_post_file}'.")

    output_file = outline_file.replace(".txt", "_draft.md")
    save_file(output_file, blog_post_draft)
    print(f"Blog post draft saved to '{output_file}'.")

    send_slack_notification(f"New blog post draft generated!")


if __name__ == "__main__":
    main()

In [ ]:
import base64, sys, os, concurrent.futures, requests

# Run "uv sync" to install the below packages
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)

client = OpenAI()


def load_file(path: str) -> str:
    if not os.path.exists(path):
        print(f"Error: The file '{path}' does not exist.")
        sys.exit(1)

    print("Loading file:", path)
    with open(path, "r", encoding="utf-8") as file:
        return file.read()


def save_file(path: str, content: str) -> None:
    print("Saving file:", path)
    with open(path, "w", encoding="utf-8") as file:
        file.write(content)


def load_and_format_example_posts(
    path: str, allowed_extensions: list[str], required: bool = False
) -> str:
    """Loads and formats example posts from a directory."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"The directory '{path}' does not exist.")

    example_posts = []
    for filename in os.listdir(path):
        if any(filename.lower().endswith(ext) for ext in allowed_extensions):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
                example_posts.append(file.read())

    if required and not example_posts:
        raise ValueError(
            f"No example posts with extensions {allowed_extensions} found in the '{path}' directory."
        )

    return "\n\n".join(
        f"<example-post-{i+1}>\n{post}\n</example-post-{i+1}>"
        for i, post in enumerate(example_posts)
    )


def generate_article_draft(
    outline: str, existing_draft: str | None = None, feedback: str | None = None
) -> str:
    print("Generating article draft...")
    example_posts_str = load_and_format_example_posts(
        "example_posts", [".md", ".mdx"], required=True
    )

    prompt = f"""
                Write a detailed blog post based on the following outline:

                <outline>
                {outline}
                </outline>

                Below are some example blog posts I wrote in the past:
                <example-posts>
                {example_posts_str}
                </example-posts>

                Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
                DON'T use the content from those example posts!

                Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
                Don't add any additional text or explanations, just return the raw markdown content.
            """

    if existing_draft and feedback:
        example_posts_str += (
            f"\n\n<existing-draft>\n{existing_draft}\n</existing-draft>"
        )
        example_posts_str += f"\n\n<feedback>\n{feedback}\n</feedback>"

        prompt = f"""
            Write an improved version of the following blog post draft:

            <existing-draft>
            {existing_draft}
            </existing-draft>

            The following feedback should be taken into account when writing the improved draft:

            <feedback>
            {feedback}
            </feedback>

            The original draft AND your improved version should be based on the following outline:

            <outline>
            {outline}
            </outline>

            Below are some example blog posts I wrote in the past:
            <example-posts>
            {example_posts_str}
            </example-posts>

            Use the language, tone, style and way of writing from the example posts to generate your draft for the new blog post.
            DON'T use the content from those example posts!

            Return the blog post draft in raw markdown format so that I can directly use it in my markdown-processing pipeline.
            Don't add any additional text or explanations, just return the raw markdown content.
        """

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post author who excels at writing engaging educational blog posts.
                    Avoid using marketing language or jargon.
                    Write in a clear, concise, and informative style for an audience that comprises both technical and non-technical readers.
                    The blog post should be structured, informative, and easy to read.

                    If the user provides an existing draft, use it in conjunction with the provided outline and feedback to create an improved draft.
                """,
            },
            {"role": "user", "content": prompt},
        ],
    )

    generated_text = response.output_text

    if generated_text.strip().startswith("```markdown"):
        lines = generated_text.strip().splitlines()
        if len(lines) > 2 and lines[-1].strip() == "```":
            generated_text = "\n".join(lines[1:-1])

    return generated_text


class Evaluation(BaseModel):
    needs_improvement: bool = Field(
        description="Whether the draft needs to be improved"
    )
    feedback: str = Field(description="Feedback on how to improve the draft")


def evaluate_article_draft(draft: str) -> Evaluation:
    print("Evaluating article draft...")
    response = client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert blog post evaluator.
                    Your job is to evaluate the quality of the blog post draft and provide feedback on how to improve it.
                    The blog post draft is in raw markdown format and based on an outline provided by the user.
                    Take that outline into account when evaluating the draft.
                    The draft should be evaluated based on the following criteria:
                    - Is the draft informative and easy to read?
                    - Is the draft structured and well-organized?
                    - Is the draft written in a clear, concise, and engaging style?
                    - Is the draft written in a way that is easy to understand for both technical and non-technical readers?
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Evaluate the following blog post draft:
                    <draft>
                    {draft}
                    </draft>

                    Return the feedback as JSON, indicating whether the draft needs to be improved and why.
                """,
            },
        ],
        text_format=Evaluation,
    )

    return response.output_parsed


def generate_thumbnail(article: str) -> bytes:
    print("Generating thumbnail...")

    response = client.images.generate(
        model="gpt-image-1",
        prompt=f"Generate a thumbnail for the following blog post: {article}",
        n=1,
        output_format="jpeg",
        size="1536x1024",
    )

    image_bytes = base64.b64decode(response.data[0].b64_json)
    return image_bytes


def generate_linkedin_post(article: str) -> str:
    print("Generating LinkedIn post...")

    example_posts_str = load_and_format_example_posts(
        "example_linkedin_posts", [".txt"]
    )

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "developer",
                "content": """
                    You are an expert LinkedIn post generator.
                    Your job is to generate a LinkedIn post for a blog post.
                    The LinkedIn post should be a single post that captures the essence of the blog post.
                    It should be informative and engaging.
                """,
            },
            {
                "role": "user",
                "content": f"""
                    Generate a LinkedIn post for the following blog post:
                    <article>
                    {article}
                    </article>

                    Here are some example LinkedIn posts I wrote in the past:
                    <example-posts>
                    {example_posts_str}
                    </example-posts>

                    Use the language, tone, style and way of writing from the example posts to generate your LinkedIn post.
                    DON'T use the content from those example posts!
                """,
            },
        ],
    )

    return response.output_text


def request_user_feedback() -> str:
    print("Your feedback on the article:")
    print("--------------------------------")
    print("Hit ENTER to accept the evaluation results without any changes.")
    print(
        "Type 'accept' to accept the article as is (and overwrite any suggested changes)."
    )
    print("Otherwise enter your feedback to improve the article.")
    print("--------------------------------")

    return input("Your feedback: ")


def send_slack_notification(message: str) -> None:
    print("Sending Slack notification...")
    response = requests.post(
        "https://slack.com/api/chat.postMessage",
        json={"channel": "C092YQF30BC", "text": message},
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ.get('SLACK_ACCESS_TOKEN')}",
        },
    )

    response.raise_for_status()
    data = response.json()

    if data.get("ok"):
        print("Slack notification sent successfully.")
    else:
        raise Exception(f"Failed to send Slack notification: {data.get('error')}")


def main():
    if len(sys.argv) != 2:
        print("Usage: python main.py <outline_file>")
        sys.exit(1)

    outline_file = sys.argv[1]
    outline = load_file(outline_file)

    blog_post_draft = None
    cycles = 0
    evaluation = None

    while True:
        blog_post_draft = generate_article_draft(
            outline,
            existing_draft=blog_post_draft,
            feedback=evaluation.feedback if evaluation else None,
        )
        print("Generated blog post draft:")
        print(blog_post_draft)

        evaluation = evaluate_article_draft(blog_post_draft)
        print("Evaluation result:")
        print(evaluation)

        user_feedback = request_user_feedback()

        if user_feedback.strip().lower() == "accept":
            evaluation.needs_improvement = False
        elif user_feedback.strip():
            evaluation.needs_improvement = True
            evaluation.feedback = user_feedback

        cycles += 1

        if not evaluation.needs_improvement or (
            cycles > 3 and not user_feedback.strip()
        ):
            break

    with concurrent.futures.ThreadPoolExecutor() as executor:
        future_thumbnail = executor.submit(generate_thumbnail, blog_post_draft)
        future_linkedin = executor.submit(generate_linkedin_post, blog_post_draft)

        linkedin_post = future_linkedin.result()
        thumbnail_image = future_thumbnail.result()

    thumbnail_file = outline_file.replace(".txt", "_thumbnail.jpeg")
    with open(thumbnail_file, "wb") as f:
        f.write(thumbnail_image)
    print(f"Thumbnail saved to '{thumbnail_file}'.")

    linkedin_post_file = outline_file.replace(".txt", "_linkedin_post.txt")
    save_file(linkedin_post_file, linkedin_post)
    print(f"LinkedIn post saved to '{linkedin_post_file}'.")

    output_file = outline_file.replace(".txt", "_draft.md")
    save_file(output_file, blog_post_draft)
    print(f"Blog post draft saved to '{output_file}'.")

    send_slack_notification(f"New blog post draft generated!")


if __name__ == "__main__":
    main()